## Limpeza dos dados: raw para cleaned
Esse notebook lê as 20 tabelas da camada `raw`, aplica 16 tratamentos de qualidade de dados (texto, número, data, chave e duplicidade) e salva o resultado na camada `cleaned`, junto com a tabela `cleaning_log`, que mostra o efeito de cada tratamento.

#### Rotina de conexão com o Neon

In [0]:
%pip install python-dotenv

In [0]:
dbutils.library.restartPython()

In [0]:
%run ../apollo-core/apollo_connection_neon_core

#### Visualizar e analisar os dados da base

##### Imports

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

##### Colunas de texto de uma tabela

In [0]:
def text_columns(df):
    return [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]

##### Perfil de uma tabela

In [0]:
def profile(df):
    types = dict(df.dtypes)
    rows = []
    for column in df.columns:
        nulls = df.filter(F.col(column).isNull()).count()
        distinct = df.select(column).distinct().count()
        rows.append((column, types[column], nulls, distinct))
    return spark.createDataFrame(rows, "coluna string, tipo string, nulos long, distintos long")

##### Rodar o perfil em uma tabela

In [0]:
df = spark.table("workspace.raw.dim_company")
display(profile(df))

##### Olhar as categorias dos textos

In [0]:
for column in text_columns(df):
    if df.select(column).distinct().count() <= 15:
        display(df.groupBy(column).count().orderBy("count", ascending=False))

#### Tratamento dos dados

Cada tratamento é uma função. Todos recebem o DataFrame (e, quando precisam, o nome da tabela) e devolvem o DataFrame tratado. Quando a tabela não tem a coluna que o tratamento procura, a função devolve o mesmo DataFrame, sem custo.

##### Imports dos tratamentos

In [0]:
import re
from pyspark.sql.types import DoubleType, FloatType, NumericType, DateType, TimestampType
from pyspark.sql.window import Window

##### Configurações dos tratamentos

Tudo que depende do domínio do Apollo fica aqui. Quando o dataload sujo chegar e aparecerem erros novos, é só ajustar esta célula.

In [0]:
FAKE_NULLS = ["", "s/n", "n/a", "na", "n.a.", "null", "none", "nan", "nil", "-", "--", "?", "nulo",
              "sem informacao", "sem informação", "nao informado", "não informado", "#n/a"]

CATEGORY_PATTERN = r"(status|stats|severity|scope|state|warning_type)$"

DOMAINS = {
    ("fact_maintenance", "maintenance_status"): ["ABERTA", "EM_ANDAMENTO", "CONCLUIDA", "CANCELADA"],
    ("fact_warning", "maintenance_status"): ["ABERTA", "EM_ANDAMENTO", "CONCLUIDA", "CANCELADA"],
    ("fact_warning", "status"): ["ATIVO", "RECONHECIDO", "RESOLVIDO"],
    ("fact_relocation", "status"): ["PENDENTE", "APROVADA", "REJEITADA", "CONCLUIDA"],
    ("dim_panel", "operating_stats"): ["EM_ESTOQUE", "OPERACIONAL", "MANUTENCAO", "INATIVO", "FALHA"],
}

NEGATIVE_PATTERN = r"(energy|kwh|power|cost|quantity|stock|days|hours|count|kwp|value|warranty)"

RANGES = {
    "health_score": (0, 100),
    "peak_to_nominal_ratio": (0, 1.5), 
    "warranty_years": (0, 50),
}

MIN_DATE = "2000-01-01"                                 
FUTURE_EXEMPT = r"(due|deadline|expected|scheduled|planned|expir)"  
DATE_EXEMPT_TABLES = ["dim_date"]                        

CHRONOLOGY = [("dim_panel", "installation_dt", "deactivated_at")]

ORDER_COLS = ["updated_at", "synced_at", "calculated_at"]

FOREIGN_KEYS = {
    "dim_company_unit": [("company_key", "dim_company")],
    "dim_inverter": [("company_unit_key", "dim_company_unit")],
    "dim_string": [("inverter_key", "dim_inverter"), ("panel_model_key", "dim_panel_model")],
    "dim_panel": [("string_key", "dim_string")],
    "fact_string_daily_performance": [("string_key", "dim_string"), ("date_key", "dim_date")],
    "fact_string_health": [("string_key", "dim_string")],
    "fact_maintenance": [("company_unit_key", "dim_company_unit"), ("maintenance_type_key", "dim_maintenance_type")],
    "fact_warning": [("company_unit_key", "dim_company_unit")],
    "fact_stock": [("company_unit_key", "dim_company_unit"), ("part_key", "dim_part")],
}

#### Tratamentos de texto

##### 1. Espaços extras

In [0]:
def trim_texts(df):
    for column in text_columns(df):
        df = df.withColumn(column, F.trim(F.regexp_replace(F.col(column), r"\s+", " ")))
    return df

##### 2. Nulo escrito como texto

In [0]:
def fake_null_to_null(df, table=None):
    for column in text_columns(df):
        df = df.withColumn(column, F.when(F.lower(F.col(column)).isin(FAKE_NULLS), None).otherwise(F.col(column)))
    return df

##### 3. Caixa das categorias

In [0]:
def category_columns(df):
    return [column for column in text_columns(df) if re.search(CATEGORY_PATTERN, column)]

def upper_categories(df):
    for column in category_columns(df):
        df = df.withColumn(column, F.upper(F.col(column)))
    return df

##### 4. Grafia das categorias

In [0]:
ACCENTS_FROM = "ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇ"
ACCENTS_TO = "AAAAAEEEEIIIIOOOOOUUUUC"
assert len(ACCENTS_FROM) == len(ACCENTS_TO)

def fix_category_spelling(df):
    for column in category_columns(df):
        if re.search(r"state$", column):  
            continue
        value = F.translate(F.col(column), ACCENTS_FROM, ACCENTS_TO)
        df = df.withColumn(column, F.regexp_replace(value, r"[\s\-]+", "_"))
    return df

##### 5. Categoria fora do domínio

In [0]:
def invalid_domain_to_null(df, table):
    for (domain_table, column), valid in DOMAINS.items():
        if domain_table == table and column in df.columns:
            df = df.withColumn(column, F.when(F.col(column).isin(valid), F.col(column)))
    return df

#### Tratamentos numéricos

##### 6. NaN e infinito

In [0]:
def nan_to_null(df):
    for field in df.schema.fields:
        if isinstance(field.dataType, (DoubleType, FloatType)):
            column = field.name
            bad = F.isnan(column) | F.col(column).isin(float("inf"), float("-inf"))
            df = df.withColumn(column, F.when(bad, None).otherwise(F.col(column)))
    return df

##### 7. Negativo impossível

In [0]:
def negative_to_null(df):
    for field in df.schema.fields:
        column = field.name
        if (isinstance(field.dataType, NumericType) and re.search(NEGATIVE_PATTERN, column)
                and not re.search(r"(_key$|_id$|delta)", column)):
            df = df.withColumn(column, F.when(F.col(column) < 0, None).otherwise(F.col(column)))
    return df

##### 8. Fora da faixa

In [0]:
def out_of_range_to_null(df):
    for column, (low, high) in RANGES.items():
        if column in df.columns:
            df = df.withColumn(column, F.when(F.col(column).between(low, high), F.col(column)))
    return df

##### 9. Outlier extremo de geração

In [0]:
def extreme_outlier_to_null(df, table):
    if table != "fact_string_daily_performance" or "generated_energy_kwh" not in df.columns:
        return df
    q = (df.groupBy("string_key")
           .agg(F.percentile_approx("generated_energy_kwh", [0.25, 0.75]).alias("q"))
           .select("string_key", F.col("q")[0].alias("_q1"), F.col("q")[1].alias("_q3")))
    df = df.join(q, "string_key", "left")
    iqr = F.col("_q3") - F.col("_q1")
    extreme = (iqr > 0) & (F.col("generated_energy_kwh") > F.col("_q3") + 3 * iqr)
    df = df.withColumn("generated_energy_kwh", F.when(extreme, None).otherwise(F.col("generated_energy_kwh")))
    return df.drop("_q1", "_q3")

#### Tratamentos de data

##### 10. Data no futuro

In [0]:
def date_columns(df):
    return [f.name for f in df.schema.fields if isinstance(f.dataType, (DateType, TimestampType))]

def future_dates_to_null(df, table):
    if table in DATE_EXEMPT_TABLES:
        return df
    limit = F.current_timestamp() + F.expr("INTERVAL 1 DAY")
    for column in date_columns(df):
        if not re.search(FUTURE_EXEMPT, column):
            df = df.withColumn(column, F.when(F.col(column) > limit, None).otherwise(F.col(column)))
    return df

##### 11. Data de preenchimento padrão

In [0]:
def placeholder_dates_to_null(df, table):
    if table in DATE_EXEMPT_TABLES:
        return df
    for column in date_columns(df):
        df = df.withColumn(column, F.when(F.col(column) < F.lit(MIN_DATE).cast("date"), None).otherwise(F.col(column)))
    return df

##### 12. Ordem cronológica

In [0]:
def invalid_chronology(df, table):
    for chrono_table, first, second in CHRONOLOGY:
        if chrono_table == table and first in df.columns and second in df.columns:
            df = df.withColumn(first, F.when(F.col(second) < F.col(first), None).otherwise(F.col(first)))
    return df

#### Tratamentos de chave e duplicidade

##### 14. Chave nula

In [0]:
def drop_null_keys(df, keys):
    return df.dropna(subset=keys) if keys else df

##### 13. Linha duplicada

In [0]:
def drop_exact_duplicates(df):
    return df.dropDuplicates()

##### 15. Chave duplicada

In [0]:
def drop_key_duplicates(df, keys):
    if not keys:
        return df
    order = next((c for c in ORDER_COLS if c in df.columns and c not in keys), None)
    if order is None:
        return df.dropDuplicates(keys)
    w = Window.partitionBy(*keys).orderBy(F.col(order).desc_nulls_last())
    return df.withColumn("_rn", F.row_number().over(w)).filter(F.col("_rn") == 1).drop("_rn")

##### 16. Chave estrangeira órfã

In [0]:
def drop_orphans(df, table):
    for column, parent in FOREIGN_KEYS.get(table, []):
        if column in df.columns:
            parent_keys = spark.table(f"{CATALOG}.cleaned.{parent}").select(F.col(column).alias("_pk")).distinct()
            df = df.join(parent_keys, df[column] == parent_keys["_pk"], "left")
            df = df.filter(F.col(column).isNull() | F.col("_pk").isNotNull()).drop("_pk")
    return df

#### Chaves de cada tabela

##### Chaves por tabela

In [0]:
TABLE_KEYS = {
    "dim_company": ["company_key"],
    "dim_segment": ["segment_key"],
    "dim_company_unit": ["company_unit_key"],
    "dim_panel_model": ["panel_model_key"],
    "dim_inverter": ["inverter_key"],
    "dim_string": ["string_key"],
    "dim_panel": ["panel_key"],
    "dim_employee": ["employee_key"],
    "dim_date": ["date_key"],
    "dim_maintenance_type": ["maintenance_type_key"],
    "dim_part": ["part_key"],
    "fact_string_daily_performance": ["string_key", "date_key"],
    "fact_string_health": ["string_key", "calculated_at"],
    "fact_maintenance": ["maintenance_key"],
    "fact_part_consumption": ["maintenance_key", "part_key"],
    "fact_daily_active_user": ["company_key", "date_key"],
    "fact_warning": ["warning_key"],
    "fact_relocation": ["relocation_scope", "relocation_key"],
    "fact_stock": ["company_unit_key", "part_key"],
    "vw_maintenance_chain": ["id"],
}

##### Ordem das tabelas

In [0]:
TABLE_ORDER = [
    "dim_company", "dim_segment", "dim_company_unit", "dim_panel_model", "dim_inverter", "dim_string",
    "dim_panel", "dim_employee", "dim_date", "dim_maintenance_type", "dim_part",
    "fact_string_daily_performance", "fact_string_health", "fact_maintenance", "fact_part_consumption",
    "fact_daily_active_user", "fact_warning", "fact_relocation", "fact_stock", "vw_maintenance_chain",
]
assert sorted(TABLE_ORDER) == sorted(VIEWS), "TABLE_ORDER e VIEWS precisam ter as mesmas tabelas"

#### Salvar as tabelas na cleaned

##### Ordem dos tratamentos

A ordem importa: primeiro padroniza o texto, depois zera o que é inválido, e só então remove linhas (chave nula, duplicadas e órfãs), para não perder linha que ficaria boa depois da padronização.

In [0]:
STEPS = [
    ("1 espacos extras",           lambda df, t: trim_texts(df), None),
    ("2 nulo escrito como texto",  lambda df, t: fake_null_to_null(df), 'valor'),
    ("3 caixa das categorias",     lambda df, t: upper_categories(df), None),
    ("4 grafia das categorias",    lambda df, t: fix_category_spelling(df), None),
    ("5 categoria fora do dominio", lambda df, t: invalid_domain_to_null(df, t), 'valor'),
    ("6 nan e infinito",           lambda df, t: nan_to_null(df), 'valor'),
    ("7 negativo impossivel",      lambda df, t: negative_to_null(df), 'valor'),
    ("8 fora da faixa",            lambda df, t: out_of_range_to_null(df), 'valor'),
    ("9 outlier extremo",          lambda df, t: extreme_outlier_to_null(df, t), 'valor'),
    ("10 data no futuro",          lambda df, t: future_dates_to_null(df, t), 'valor'),
    ("11 data padrao antiga",      lambda df, t: placeholder_dates_to_null(df, t), 'valor'),
    ("12 ordem cronologica",       lambda df, t: invalid_chronology(df, t), 'valor'),
    ("13 linha duplicada",         lambda df, t: drop_exact_duplicates(df), 'linha'),
    ("14 chave nula",              lambda df, t: drop_null_keys(df, TABLE_KEYS.get(t, [])), 'linha'),
    ("15 chave duplicada",         lambda df, t: drop_key_duplicates(df, TABLE_KEYS.get(t, [])), 'linha'),
    ("16 chave estrangeira orfa",  lambda df, t: drop_orphans(df, t), 'linha'),
]

##### Medir o efeito de cada tratamento

In [0]:
def snapshot(df):
    nulls = sum((F.col(c).isNull().cast("int") for c in df.columns), F.lit(0))
    row = df.agg(F.count("*").alias("linhas"), F.sum(nulls).alias("nulos")).first()
    return row["linhas"], int(row["nulos"] or 0)

##### Tratar uma tabela

In [0]:
def clean_table(df, table, log):
    rows, nulls = snapshot(df)
    for name, step, effect in STEPS:
        new_df = step(df, table)
        if new_df is df:             
            continue
        if effect:
            new_rows, new_nulls = snapshot(new_df)
            removed = rows - new_rows if effect == "linha" else 0
            created = new_nulls - nulls if effect == "valor" else 0
            log.append((table, name, removed, created))
            rows, nulls = new_rows, new_nulls
        df = new_df
    return df

##### Rodar o tratamento e salvar cada tabela

In [0]:
log = []

for table in TABLE_ORDER:
    raw_df = spark.table(f"{CATALOG}.raw.{table}")
    cleaned_df = clean_table(raw_df, table, log)
    cleaned_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.cleaned.{table}")
    print(table, "ok:", raw_df.count() - spark.table(f"{CATALOG}.cleaned.{table}").count(), "linhas removidas")

##### Salvar o log dos tratamentos

In [0]:
log_df = (spark.createDataFrame(log, "tabela string, tratamento string, linhas_removidas long, nulos_criados long")
          .filter("linhas_removidas != 0 OR nulos_criados != 0")
          .withColumn("executado_em", F.current_timestamp()))
log_df.write.mode("overwrite").saveAsTable(f"{CATALOG}.cleaned.cleaning_log")
display(log_df.orderBy("tabela", "tratamento"))

#### Conferir as tabelas

In [0]:
display(spark.sql(f"SHOW TABLES IN {CATALOG}.cleaned"))